In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample2") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

# Устанавливаем таймауты и keep-alive как числа (без 's')
# Значения в секундах или миллисекундах (зависит от версии, обычно keepalivetime в сек)
hadoop_conf.set("fs.s3a.threads.keepalivetime", "60") 
hadoop_conf.set("fs.s3a.connection.timeout", "60000")
hadoop_conf.set("fs.s3a.attempts.maximum", "10")
hadoop_conf.set("fs.s3a.connection.establish.timeout", "5000")
hadoop_conf.set("fs.s3a.readahead.range", "65536")

hadoop_conf.set("fs.s3a.multipart.purge.age", "86400")

hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")



:: loading settings :: url = jar:file:/home/coder/alex_env/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-12efe607-903d-4c13-80db-2f6941763237;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clic

In [2]:
import os
# ⬇️ Параметры подключения к PostgreSQL
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

base_jdbc_options = {
    "url": jdbc_url,
    "user": db_user,
    "password": db_password,
    "fetchsize": 1000,
    "driver": "org.postgresql.Driver"
}

shops_df = spark.read \
            .format("jdbc") \
            .options(**base_jdbc_options) \
            .option("dbtable", "public.shops") \
            .load()

shop_tz_df = spark.read \
            .format("jdbc") \
            .options(**base_jdbc_options)\
            .option("dbtable", "public.shop_timezone") \
            .load()

# Регистрируем DataFrame-ы как временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_tz_df.createOrReplaceTempView("shop_timezone")

In [3]:
from pyspark.sql.functions import col, when, substring

sz =(
        shop_tz_df
        .where(""" try_cast(plant as INT) IS NOT NULL and time_zone != '' """)
        .select(
            col("plant").try_cast("int").alias("id"),
            "time_zone")
    )
                
s = shops_df.select(col("st_id").cast("int").alias("st_id"), "shop_name")


from pyspark.sql.functions import row_number
from pyspark.sql import Window
window_spec = Window.partitionBy("st_id").orderBy("st_id")

cte = sz \
        .join(s, sz.id==s.st_id, "right") \
        .select("*") \
        .withColumn("rank", row_number().over(window_spec))


final_df = (
        cte
        .where(""" rank = 1 """)
        .select("st_id",
                "shop_name",
                "time_zone")
        .withColumn("tz_code",
                when(col("time_zone").isNull(), 3)
                .otherwise(substring(col("time_zone"), 4, 10).cast("int")))
        .drop("time_zone")
)

In [4]:
# ⬇️ Параметры подключения к CLICKHOUSE
jdbc_url = 'jdbc:clickhouse://clickhouse01:8123/alexxxxxxela'
db_user = os.getenv('CLICKHOUSE_USER')
db_password = os.getenv('CLICKHOUSE_PASSWORD')
table_name = 'shops_local'

final_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("truncate", "true") \
    .mode("append") \
    .save()

print("Таблица сохранена в Clickhouse")

Таблица сохранена в Clickhouse


26/01/29 08:28:31 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/01/29 08:28:31 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/01/29 08:28:31 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [544303a7-9753-4395-8332-5a35dc7e5797](2 queries & 0 savepoints) is committed.
26/01/29 08:28:31 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [90dd8500-06bc-468d-a665-f5c0a291ee2d](0 queries & 0 savepoints) is committed.
